# 6. The repository's own verification suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/v40.30/notebooks/06_verification_suite.ipynb)

Runs `python verify.py` (the fast path, about 6 minutes plus about 1 minute of installation): table and prose recomputation, regression tests, certificates and the file manifest. The core verifier also regenerates the figures and compares them byte for byte, which needs the exact library versions of `requirements-lock.txt`. The notebook therefore runs the suite inside a small dedicated environment with those versions, leaving your own Python installation unchanged. The longer suites are `python verify.py --full` and `--everything`; the Lean proofs need a Lean toolchain (see `LEAN_VERIFICATION.md`).

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

Each step is labelled **[recompute from source]** (values derived again from the data or from first principles), **[re-run an experiment]** (a repository script is executed) or **[compare against stored output]** (a committed result file is checked against the paper).

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the pinned revision first).
import os, sys, subprocess, platform
REVISION = os.environ.get("LOYOLA_REVISION", "v40.30")
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REVISION, "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:   # pinned versions, as used for the recorded results
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-lock.txt", "-r", "requirements-extras-lock.txt"], check=True)
ROOT = os.getcwd()
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
HEAD, DIRTY = git("rev-parse", "HEAD"), bool(git("status", "--porcelain", "--untracked-files=no"))
PINNED = git("rev-parse", "--verify", "--quiet", REVISION + "^{commit}")
if HEAD and not PINNED:   # shallow or local clone without the tag: try to fetch it before comparing
    subprocess.run(["git", "fetch", "--quiet", "--depth", "1", "origin", "tag", REVISION], capture_output=True)
    PINNED = git("rev-parse", "--verify", "--quiet", REVISION + "^{commit}")
print(f"commit {HEAD or 'unknown (no git metadata)'}{' (uncommitted changes)' if DIRTY else ''}; "
      f"pinned revision {REVISION}: {PINNED or 'could not be resolved'}")
if (not HEAD or not PINNED or PINNED != HEAD or DIRTY) and os.environ.get("LOYOLA_ALLOW_MISMATCH") != "1":
    raise RuntimeError(f"REVISION MISMATCH: cannot confirm that the files are {REVISION} "
                       f"(checked out: {HEAD or 'unknown'}{' with uncommitted changes' if DIRTY else ''}; "
                       f"{REVISION}: {PINNED or 'unresolved'}). "
                       "Check out the pinned revision, or set LOYOLA_ALLOW_MISMATCH=1 to run anyway.")
if os.environ.get("LOYOLA_ALLOW_MISMATCH") == "1" and (not PINNED or PINNED != HEAD or DIRTY):
    print("WARNING: LOYOLA_ALLOW_MISMATCH=1 - these results are NOT tied to the pinned revision")
import numpy, scipy, networkx, mpmath
print("python", platform.python_version(), "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| networkx", networkx.__version__, "| mpmath", mpmath.__version__)
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

commit 09ceba8bd43baff4eec440dff1d1b860d563d948; pinned revision v40.30: could not be resolved


python 3.13.16 | numpy 2.4.4 | scipy 1.17.1 | networkx 3.6.1 | mpmath 1.3.0
working directory: /home/mpcrlab/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
import venv
env = os.path.join(ROOT, ".venv-verify")
py = os.path.join(env, "bin", "python")
if not os.path.exists(py):
    venv.create(env, with_pip=True)
    run(f"{py} -m pip install -q -r requirements-lock.txt", tail=3)
run(f"{py} -m pip list --format=freeze", tail=20)

contourpy==1.3.3
cycler==0.12.1
fonttools==4.62.1
kiwisolver==1.5.0
matplotlib==3.10.8
mpmath==1.3.0
networkx==3.6.1
numpy==2.4.4
packaging==26.0
pillow==12.2.0
pip==26.2.1
pyparsing==3.3.2
python-dateutil==2.9.0.post0
scipy==1.17.1
six==1.17.0
sympy==1.14.0
[exit 0, 2 s]


'contourpy==1.3.3\ncycler==0.12.1\nfonttools==4.62.1\nkiwisolver==1.5.0\nmatplotlib==3.10.8\nmpmath==1.3.0\nnetworkx==3.6.1\nnumpy==2.4.4\npackaging==26.0\npillow==12.2.0\npip==26.2.1\npyparsing==3.3.2\npython-dateutil==2.9.0.post0\nscipy==1.17.1\nsix==1.17.0\nsympy==1.14.0\n'

In [4]:
out = run(f"{py} verify.py", tail=40)
check("fast path result", True, "ALL PASS" in out)

[FAIL] core verifier (1720 s): math failures: 0
[PASS] prose numbers (112 s): prose numbers: 742 checked, 0 failures, 20 uncheckable (cited values; see UNCHECKABLE in the registries)
[PASS] external regression tests (4 s): OK
[PASS] baseline tests (1675 s): 0 failures; 1673.6s
[PASS] constancy test check (704 s): RESULT: the two constancy tests never disagree on these inputs; no fold is affected
[PASS] auxiliary outputs (18 s): Auxiliary output reproduction PASS
[PASS] archived-source identities (1 s): Archived source formula/skeleton checks PASS: 154 pages
[PASS] interval certificates (6 s): Mathematics interval checks PASS
[PASS] prose style (0 s): prose style check: PASS
[PASS] extremal trees (1 s): RESULT: extremal-tree proposition holds for all orders 4..12
[PASS] collision minimality (15 s): RESULT: first order with distinct profiles sharing a q-histogram: 13 (all trees), 16 (maximum degree <= 4)
[PASS] sum-connectivity pair (14 s): RESULT: Example ex:sumconn confirmed (D(1) = 0.

False

## Summary

In [5]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                                                                    check   paper recomputed    status
0  script: /home/mpcrlab/LOYOLA-PAPER/.venv-verify/bin/python -m pip list --format=freeze  exit 0     exit 0        OK
1                    script: /home/mpcrlab/LOYOLA-PAPER/.venv-verify/bin/python verify.py  exit 0     exit 1  MISMATCH
2                                                                        fast path result    True      False  MISMATCH
3 checks, 2 mismatches


AssertionError: some recomputed values differ from the paper